# Nemotron on Amazon Bedrock: RAG with Knowledge Bases and Guardrails

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline using
NVIDIA Nemotron models on Amazon Bedrock, combined with **Bedrock Knowledge Bases** for document
retrieval and **Bedrock Guardrails** for responsible AI output.

We use publicly available mortgage and home-buying guides from HUD and the CFPB as our knowledge source.

We will programmatically:
1. Create an S3 bucket and upload PDF documents
2. Create an IAM role for the Knowledge Base
3. Set up an Amazon OpenSearch Serverless vector store
4. Create a Bedrock Knowledge Base with the Amazon Titan Text Embeddings V2 model
5. Ingest documents into the Knowledge Base
6. Create a Bedrock Guardrail with content and topic filters
7. Query the Knowledge Base with `retrieve_and_generate`, applying the Guardrail

## Prerequisites

- An AWS account with access to Amazon Bedrock, Amazon OpenSearch Serverless, and Amazon S3
- Model access enabled for **NVIDIA Nemotron** and **Amazon Titan Text Embeddings V2** in the Bedrock console
- IAM permissions to create roles, policies, S3 buckets, and OpenSearch Serverless collections
- Python 3.10+

## Step 1: Environment Setup

In [ ]:
%pip install -U boto3 opensearch-py requests-aws4auth requests

In [ ]:
import boto3
import json
import time
import uuid
import os
import requests
from botocore.exceptions import ClientError

In [ ]:
import os
from pathlib import Path
if not Path("pdf_data").exists():
    os.chdir(next(Path.home().rglob("pdf_data")).parent)
print(f"cwd: {Path.cwd()}")

### Configuration

Set the AWS region and resource names. Update these values as needed for your environment.

In [ ]:
# ── AWS Region ──
REGION = "us-west-2"

# ── Model IDs ──
NEMOTRON_MODEL_ID = "nvidia.nemotron-nano-3-30b"  # Generation model
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"     # Embedding model for Knowledge Base

# ── Resource naming ──
SUFFIX = str(uuid.uuid4())[:8]
S3_BUCKET_NAME = f"nemotron-kb-data-{SUFFIX}"
KB_NAME = f"nemotron-rag-kb-{SUFFIX}"
GUARDRAIL_NAME = f"nemotron-guardrail-{SUFFIX}"
AOSS_COLLECTION_NAME = f"nemotron-kb-{SUFFIX}"  # Must be lowercase, max 32 chars
KB_ROLE_NAME = f"AmazonBedrockKBRole-{SUFFIX}"
INDEX_NAME = "bedrock-kb-index"
EMBEDDING_DIMENSION = 1024  # Amazon Titan Text Embeddings V2 output dimension

# ── Derived ARNs ──
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
EMBEDDING_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL_ID}"
NEMOTRON_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/{NEMOTRON_MODEL_ID}"

print(f"Account ID:       {ACCOUNT_ID}")
print(f"Region:           {REGION}")
print(f"S3 Bucket:        {S3_BUCKET_NAME}")
print(f"KB Name:          {KB_NAME}")
print(f"Guardrail Name:   {GUARDRAIL_NAME}")
print(f"AOSS Collection:  {AOSS_COLLECTION_NAME}")
print(f"Embedding Model:  {EMBEDDING_MODEL_ARN}")
print(f"Generation Model: {NEMOTRON_MODEL_ARN}")

### Initialize AWS Clients

In [ ]:
s3_client = boto3.client("s3", region_name=REGION)
iam_client = boto3.client("iam")
bedrock_client = boto3.client("bedrock", region_name=REGION)
bedrock_agent_client = boto3.client("bedrock-agent", region_name=REGION)
bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime", region_name=REGION)
aoss_client = boto3.client("opensearchserverless", region_name=REGION)

print("All clients initialized.")

## Step 2: Create S3 Bucket and Upload Documents

We create an S3 bucket and upload publicly available mortgage and home-buying PDF guides from
HUD and the CFPB to S3 for the Knowledge Base to index. The PDFs are bundled in the `pdf_data/`
directory next to this notebook — make sure you run the notebook from that same directory.

In [ ]:
# Create S3 bucket
if REGION == "us-east-1":
    s3_client.create_bucket(Bucket=S3_BUCKET_NAME)
else:
    s3_client.create_bucket(
        Bucket=S3_BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION}
    )
print(f"Created S3 bucket: {S3_BUCKET_NAME}")

# Enforce HTTPS-only access by denying any request that doesn't use TLS
https_only_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "DenyInsecureTransport",
            "Effect": "Deny",
            "Principal": "*",
            "Action": "s3:*",
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET_NAME}",
                f"arn:aws:s3:::{S3_BUCKET_NAME}/*"
            ],
            "Condition": {"Bool": {"aws:SecureTransport": "false"}}
        }
    ]
}

s3_client.put_bucket_policy(
    Bucket=S3_BUCKET_NAME,
    Policy=json.dumps(https_only_policy)
)
print(f"Applied HTTPS-only bucket policy to {S3_BUCKET_NAME}")

In [ ]:
# Upload local PDF documents to S3.
# The PDFs live in the `pdf_data/` directory next to this notebook.
PDF_DIR = os.path.join(os.getcwd(), "pdf_data")

pdf_files = sorted(f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf"))

for filename in pdf_files:
    local_path = os.path.join(PDF_DIR, filename)
    print(f"Uploading {filename}...")
    with open(local_path, "rb") as f:
        content = f.read()

    s3_client.put_object(
        Bucket=S3_BUCKET_NAME,
        Key=f"documents/{filename}",
        Body=content,
        ContentType="application/pdf"
    )
    print(f"  Uploaded to s3://{S3_BUCKET_NAME}/documents/{filename} ({len(content):,} bytes)")

print(f"\nAll documents uploaded to s3://{S3_BUCKET_NAME}/documents/")

## Step 3: Create IAM Role for Knowledge Base

The Knowledge Base needs an IAM role that grants it permission to:
- Invoke the embedding model on Bedrock
- Read documents from the S3 bucket
- Read and write to the OpenSearch Serverless collection

In [ ]:
# Trust policy allowing Bedrock to assume this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
                "ArnLike": {
                    "aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/*"
                }
            }
        }
    ]
}

# Create the role
try:
    role_response = iam_client.create_role(
        RoleName=KB_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for Bedrock Knowledge Base to access S3, AOSS, and embedding model"
    )
    KB_ROLE_ARN = role_response["Role"]["Arn"]
    print(f"Created IAM role: {KB_ROLE_ARN}")
except iam_client.exceptions.EntityAlreadyExistsException:
    KB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{KB_ROLE_NAME}"
    print(f"Role already exists: {KB_ROLE_ARN}")

In [ ]:
# Inline policy granting S3, Bedrock, and AOSS permissions
kb_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET_NAME}",
                f"arn:aws:s3:::{S3_BUCKET_NAME}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": f"arn:aws:bedrock:{REGION}::foundation-model/*"
        },
        {
            "Effect": "Allow",
            "Action": "aoss:APIAccessAll",
            "Resource": f"arn:aws:aoss:{REGION}:{ACCOUNT_ID}:collection/*"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=KB_ROLE_NAME,
    PolicyName="BedrockKBPolicy",
    PolicyDocument=json.dumps(kb_policy)
)
print("Attached inline policy to role.")

# Allow time for IAM propagation (cross-service policies can take up to 60s)
print("Waiting for IAM role propagation...")
time.sleep(60)
print("Done.")

## Step 4: Create OpenSearch Serverless Vector Store

Amazon OpenSearch Serverless provides a managed vector search engine. We need to create:
1. **Encryption policy** - Encryption at rest for the collection
2. **Network policy** - Network access rules
3. **Data access policy** - Who can read/write data
4. **Collection** - The actual vector store
5. **Vector index** - Index with the correct mapping for embeddings

In [ ]:
# 1. Create encryption policy
encryption_policy = {
    "Rules": [
        {
            "ResourceType": "collection",
            "Resource": [f"collection/{AOSS_COLLECTION_NAME}"]
        }
    ],
    "AWSOwnedKey": True
}

aoss_client.create_security_policy(
    name=f"{AOSS_COLLECTION_NAME}-enc",
    type="encryption",
    policy=json.dumps(encryption_policy),
    description="Encryption policy for KB collection"
)
print("Created encryption policy.")

# 2. Create network policy
network_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{AOSS_COLLECTION_NAME}"]
            },
            {
                "ResourceType": "dashboard",
                "Resource": [f"collection/{AOSS_COLLECTION_NAME}"]
            }
        ],
        "AllowFromPublic": True
    }
]

aoss_client.create_security_policy(
    name=f"{AOSS_COLLECTION_NAME}-net",
    type="network",
    policy=json.dumps(network_policy),
    description="Network policy for KB collection"
)
print("Created network policy.")

# 3. Create data access policy
caller_arn = boto3.client("sts").get_caller_identity()["Arn"]

data_access_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{AOSS_COLLECTION_NAME}"],
                "Permission": [
                    "aoss:CreateCollectionItems",
                    "aoss:UpdateCollectionItems",
                    "aoss:DescribeCollectionItems"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{AOSS_COLLECTION_NAME}/*"],
                "Permission": [
                    "aoss:CreateIndex",
                    "aoss:UpdateIndex",
                    "aoss:DescribeIndex",
                    "aoss:ReadDocument",
                    "aoss:WriteDocument"
                ]
            }
        ],
        "Principal": [caller_arn, KB_ROLE_ARN],
        "Description": "Data access policy for KB"
    }
]

aoss_client.create_access_policy(
    name=f"{AOSS_COLLECTION_NAME}-access",
    type="data",
    policy=json.dumps(data_access_policy),
    description="Data access policy for KB collection"
)
print("Created data access policy.")

In [ ]:
# 4. Create the OpenSearch Serverless collection
collection_response = aoss_client.create_collection(
    name=AOSS_COLLECTION_NAME,
    type="VECTORSEARCH",
    description="Vector store for Nemotron RAG Knowledge Base"
)

COLLECTION_ID = collection_response["createCollectionDetail"]["id"]
print(f"Collection created: {COLLECTION_ID}")
print("Waiting for collection to become active (this may take a few minutes)...")

# Wait for collection to be active
while True:
    status = aoss_client.batch_get_collection(ids=[COLLECTION_ID])
    collection_details = status["collectionDetails"][0]
    current_status = collection_details["status"]
    print(f"  Status: {current_status}")
    if current_status == "ACTIVE":
        COLLECTION_ENDPOINT = collection_details["collectionEndpoint"]
        print(f"Collection is ACTIVE. Endpoint: {COLLECTION_ENDPOINT}")
        break
    elif current_status == "FAILED":
        raise RuntimeError("Collection creation failed.")
    time.sleep(30)

In [ ]:
# 5. Create the vector index
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    REGION,
    "aoss",
    session_token=credentials.token
)

# Connect to the collection
oss_client = OpenSearch(
    hosts=[{"host": COLLECTION_ENDPOINT.replace("https://", ""), "port": 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

# Create index with vector field mapping
index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512
        }
    },
    "mappings": {
        "properties": {
            "bedrock-knowledge-base-default-vector": {
                "type": "knn_vector",
                "dimension": EMBEDDING_DIMENSION,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw",
                    "parameters": {"ef_construction": 512, "m": 16},
                    "space_type": "l2"
                }
            },
            "AMAZON_BEDROCK_METADATA": {"type": "text", "index": False},
            "AMAZON_BEDROCK_TEXT_CHUNK": {"type": "text"}
        }
    }
}

response = oss_client.indices.create(index=INDEX_NAME, body=index_body)
print(f"Created index: {INDEX_NAME}")
print(json.dumps(response, indent=2))

# Wait for the index to be available to Bedrock. AOSS index creation acks
# immediately, but the index needs ~60s to propagate before create_knowledge_base
# can attach to it — otherwise the KB call fails with a "no such index" error.
print("Waiting 60s for index to become available to Bedrock...")
time.sleep(60)
print("Done.")

## Step 5: Create Bedrock Knowledge Base and Ingest Data

Now we create the Knowledge Base, attach an S3 data source, and start the ingestion job
to embed and index the documents.

In [ ]:
# Create the Knowledge Base
kb_response = bedrock_agent_client.create_knowledge_base(
    name=KB_NAME,
    description="Knowledge base for mortgage and home-buying information",
    roleArn=KB_ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": EMBEDDING_MODEL_ARN
        }
    },
    storageConfiguration={
        "type": "OPENSEARCH_SERVERLESS",
        "opensearchServerlessConfiguration": {
            "collectionArn": f"arn:aws:aoss:{REGION}:{ACCOUNT_ID}:collection/{COLLECTION_ID}",
            "fieldMapping": {
                "metadataField": "AMAZON_BEDROCK_METADATA",
                "textField": "AMAZON_BEDROCK_TEXT_CHUNK",
                "vectorField": "bedrock-knowledge-base-default-vector"
            },
            "vectorIndexName": INDEX_NAME
        }
    }
)

KB_ID = kb_response["knowledgeBase"]["knowledgeBaseId"]
print(f"Knowledge Base created: {KB_ID}")
print(f"Status: {kb_response['knowledgeBase']['status']}")

In [ ]:
# Wait for Knowledge Base to be active
while True:
    kb_status = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=KB_ID)
    status = kb_status["knowledgeBase"]["status"]
    print(f"  KB Status: {status}")
    if status == "ACTIVE":
        print("Knowledge Base is ACTIVE.")
        break
    elif status == "FAILED":
        raise RuntimeError(f"KB creation failed: {kb_status}")
    time.sleep(10)

In [ ]:
# Create S3 data source
ds_response = bedrock_agent_client.create_data_source(
    knowledgeBaseId=KB_ID,
    name=f"{KB_NAME}-s3-source",
    description="S3 data source with mortgage documents",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": f"arn:aws:s3:::{S3_BUCKET_NAME}",
            "inclusionPrefixes": ["documents/"]
        }
    }
)

DATA_SOURCE_ID = ds_response["dataSource"]["dataSourceId"]
print(f"Data source created: {DATA_SOURCE_ID}")

In [ ]:
# Start data ingestion job
ingestion_response = bedrock_agent_client.start_ingestion_job(
    knowledgeBaseId=KB_ID,
    dataSourceId=DATA_SOURCE_ID
)

INGESTION_JOB_ID = ingestion_response["ingestionJob"]["ingestionJobId"]
print(f"Ingestion job started: {INGESTION_JOB_ID}")
print("Waiting for ingestion to complete...")

while True:
    job_status = bedrock_agent_client.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DATA_SOURCE_ID,
        ingestionJobId=INGESTION_JOB_ID
    )
    status = job_status["ingestionJob"]["status"]
    print(f"  Ingestion status: {status}")
    if status == "COMPLETE":
        stats = job_status["ingestionJob"]["statistics"]
        print(f"Ingestion complete. Documents scanned: {stats.get('numberOfDocumentsScanned', 'N/A')}, "
              f"indexed: {stats.get('numberOfNewDocumentsIndexed', 'N/A')}, "
              f"failed: {stats.get('numberOfDocumentsFailed', 'N/A')}")
        break
    elif status == "FAILED":
        print(f"Ingestion failed: {job_status}")
        break
    time.sleep(15)

## Step 6: Create Bedrock Guardrails

Bedrock Guardrails lets you implement safeguards for your generative AI applications.
We create a guardrail with:
- **Content filters** — Block harmful content across hate, violence, sexual, and misconduct categories
- **Denied topic** — Block requests for specific investment or legal advice
- **Sensitive information filters** — Redact PII such as email addresses and phone numbers
- **Word filters** — Block specific unwanted words

In [ ]:
guardrail_response = bedrock_client.create_guardrail(
    name=GUARDRAIL_NAME,
    description="Guardrail for mortgage RAG application",
    blockedInputMessaging="Your request has been blocked by our content policy. Please rephrase your question.",
    blockedOutputsMessaging="The response has been blocked by our content policy. Please try a different question.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "SpecificInvestmentAdvice",
                "definition": "Providing specific investment recommendations, stock picks, or guarantees about financial returns.",
                "examples": [
                    "You should invest all your money in this stock",
                    "I guarantee this investment will double your money",
                    "Buy this specific property, it will appreciate 50%"
                ],
                "type": "DENY"
            },
            {
                "name": "LegalAdvice",
                "definition": "Providing specific legal advice or acting as a legal representative.",
                "examples": [
                    "You should sue your lender for this",
                    "This contract clause is illegal, ignore it",
                    "As your lawyer, I advise you to"
                ],
                "type": "DENY"
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "HATE",       "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "VIOLENCE",   "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "SEXUAL",     "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
            {"type": "INSULTS",    "inputStrength": "HIGH", "outputStrength": "HIGH"}
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"}
        ]
    },
    wordPolicyConfig={
        "wordsConfig": [
            {"text": "guaranteed returns"},
            {"text": "risk-free investment"}
        ]
    }
)

GUARDRAIL_ID = guardrail_response["guardrailId"]
GUARDRAIL_VERSION = guardrail_response["version"]
print(f"Guardrail created: {GUARDRAIL_ID} (version: {GUARDRAIL_VERSION})")

## Step 7: RAG with Knowledge Base and Guardrails

Now we bring it all together using the `retrieve_and_generate` API. This API:
1. Takes the user query and retrieves relevant documents from the Knowledge Base
2. Sends the retrieved context + query to the Nemotron model for generation
3. Applies the Guardrail to both the input and the generated output

### Query without Guardrails

First, let's query the Knowledge Base without guardrails to establish a baseline.

In [ ]:
GENERATION_PROMPT_TEMPLATE = """
You are a helpful question-answering assistant for mortgages and home buying.
Use only the information in the search results to answer the user's question.
If the search results do not contain enough information, say that you could not find
the answer in the provided sources. Provide general educational information, not legal,
investment, or financial advice.

Here are the search results in numbered order:
$search_results$

User query:
$query$

$output_format_instructions$
"""


def query_knowledge_base(query, use_guardrail=False, number_of_results=5):
    """Query the Knowledge Base with retrieve_and_generate, optionally applying a guardrail."""
    kb_config = {
        "knowledgeBaseId": KB_ID,
        "modelArn": NEMOTRON_MODEL_ARN,
        "retrievalConfiguration": {
            "vectorSearchConfiguration": {
                "numberOfResults": number_of_results
            }
        },
        "generationConfiguration": {
            "promptTemplate": {
                "textPromptTemplate": GENERATION_PROMPT_TEMPLATE
            },
            "inferenceConfig": {
                "textInferenceConfig": {
                    "maxTokens": 1024,
                    "temperature": 0.2,
                    "topP": 0.9
                }
            }
        },
        "orchestrationConfiguration": {
            "promptTemplate": {
                "textPromptTemplate": (
                    "You are very knowledgeable on mortgages and home buying. "
                    "Conversation so far:\n$conversation_history$\n\n"
                    "User query:\n$query$\n\n"
                    "$output_format_instructions$"
                )
            },
            "inferenceConfig": {
                "textInferenceConfig": {
                    "maxTokens": 512,
                    "temperature": 0.0,
                    "topP": 0.9
                }
            }
        }
    }

    if use_guardrail:
        kb_config["generationConfiguration"]["guardrailConfiguration"] = {
            "guardrailId": GUARDRAIL_ID,
            "guardrailVersion": GUARDRAIL_VERSION
        }

    return bedrock_agent_runtime_client.retrieve_and_generate(
        input={"text": query},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": kb_config
        }
    )

In [ ]:
# Query without guardrails
query = "What steps should I take to prepare for getting a mortgage?"
print(f"Query: {query}\n")

response = query_knowledge_base(query, use_guardrail=False)

print("Response:")
print(response["output"]["text"])

# Show retrieved sources
print("\n--- Retrieved Sources ---")
for i, citation in enumerate(response.get("citations", [])):
    for ref in citation.get("retrievedReferences", []):
        source = ref.get("location", {}).get("s3Location", {}).get("uri", "Unknown")
        print(f"  [{i+1}] {source}")

### Query with Guardrails

Now let's apply the guardrail to filter both input and output.

In [ ]:
# Query with guardrails enabled
query = "What steps should I take to prepare for getting a mortgage?"
print(f"Query: {query}\n")

response = query_knowledge_base(query, use_guardrail=True)

print("Response (with guardrail):")
print(response["output"]["text"])

print("\n--- Retrieved Sources ---")
for i, citation in enumerate(response.get("citations", [])):
    for ref in citation.get("retrievedReferences", []):
        source = ref.get("location", {}).get("s3Location", {}).get("uri", "Unknown")
        print(f"  [{i+1}] {source}")

### Test Guardrail Blocking

Let's test queries that should be blocked by the guardrail's topic and content filters.

In [ ]:
# Test: Denied topic - specific investment advice
blocked_queries = [
    "Tell me exactly which stocks to invest my mortgage savings in for guaranteed returns.",
    "Should I sue my mortgage lender? Give me specific legal advice on how to proceed.",
]

for query in blocked_queries:
    print(f"Query: {query}\n")
    try:
        response = query_knowledge_base(query, use_guardrail=True)
        print(f"Response: {response['output']['text']}")
    except Exception as e:
        print(f"Blocked/Error: {e}")
    print("\n" + "=" * 80 + "\n")

In [ ]:
# Test: Additional RAG queries with guardrails
rag_queries = [
    "What is the difference between a fixed-rate and adjustable-rate mortgage?",
    "When should I consider refinancing my mortgage?",
    "What credit score do I need to buy a home?",
]

for query in rag_queries:
    print(f"Query: {query}\n")
    response = query_knowledge_base(query, use_guardrail=True)
    print(f"Response: {response['output']['text']}")
    print("\n" + "-" * 80 + "\n")

## Step 8: Cleanup

Delete all resources created in this notebook to avoid ongoing charges.
Uncomment and run the cell below when you are done experimenting.

In [ ]:
# # ⚠️ Uncomment the lines below to delete all resources

# # Delete data source and knowledge base
# bedrock_agent_client.delete_data_source(knowledgeBaseId=KB_ID, dataSourceId=DATA_SOURCE_ID)
# print("Deleted data source.")
# bedrock_agent_client.delete_knowledge_base(knowledgeBaseId=KB_ID)
# print("Deleted knowledge base.")

# # Delete guardrail
# bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
# print("Deleted guardrail.")

# # Delete OpenSearch Serverless collection and policies
# aoss_client.delete_collection(id=COLLECTION_ID)
# print("Deleted AOSS collection.")
# aoss_client.delete_access_policy(name=f"{AOSS_COLLECTION_NAME}-access", type="data")
# aoss_client.delete_security_policy(name=f"{AOSS_COLLECTION_NAME}-net", type="network")
# aoss_client.delete_security_policy(name=f"{AOSS_COLLECTION_NAME}-enc", type="encryption")
# print("Deleted AOSS policies.")

# # Delete IAM role
# iam_client.delete_role_policy(RoleName=KB_ROLE_NAME, PolicyName="BedrockKBPolicy")
# iam_client.delete_role(RoleName=KB_ROLE_NAME)
# print("Deleted IAM role.")

# # Delete S3 bucket and objects
# objects = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME).get("Contents", [])
# for obj in objects:
#     s3_client.delete_object(Bucket=S3_BUCKET_NAME, Key=obj["Key"])
# s3_client.delete_bucket(Bucket=S3_BUCKET_NAME)
# print("Deleted S3 bucket.")

# print("\nAll resources cleaned up.")

## Conclusion

In this notebook, we demonstrated an end-to-end RAG pipeline using:

- **NVIDIA Nemotron** on Amazon Bedrock as the generation model
- **Amazon Titan Text Embeddings V2** as the embedding model for document ingestion
- **Bedrock Knowledge Bases** backed by OpenSearch Serverless for document retrieval
- **Bedrock Guardrails** for content filtering, topic blocking, and PII redaction

The `retrieve_and_generate` API combines these into a single RAG call, while the custom prompt
keeps Bedrock's citation output format instructions intact.

### Next Steps

- Add your own documents to the S3 bucket and re-run ingestion
- Customize the guardrail policies for your specific use case
- Explore the Bedrock console to monitor guardrail invocations and Knowledge Base metrics